Testing if T is able to pass the .5 threshold.

In [2]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRangePercentilesd, Resized,
    RandFlipd, RandRotate90d, RandGaussianNoised, EnsureTyped,
)
from sklearn.model_selection import train_test_split
import pandas as pd
from monai.data import DataLoader
import torch
from monai.networks.nets import ViT
import pydicom
pydicom.config.settings.reading_validation_mode = pydicom.config.WARN
from sklearn.preprocessing import LabelEncoder
from monai.networks.nets import ViTAutoEnc


In [3]:

df_data = pd.read_csv("./data/labels_duke_nifti.csv")
data = df_data.to_dict(orient="records")


train_data, val_data = train_test_split(
                            data, test_size=0.2, stratify=[d["label"] for d in data], random_state=42
                        )

In [4]:


transforms = Compose([
    LoadImaged(keys=["nifti"], ensure_channel_first=True),                
    Orientationd(keys="nifti", axcodes="RAS"),            
    Spacingd(keys="nifti", pixdim=(1.0, 1.0, 1.0),        
             mode="bilinear"),
    ScaleIntensityRangePercentilesd(                       
        keys="nifti", lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="nifti", spatial_size=(96, 96, 96)),      
    RandFlipd(keys="nifti", prob=0.5, spatial_axis=0),    
    RandRotate90d(keys="nifti", prob=0.5, spatial_axes=(0, 1)),
    RandGaussianNoised(keys="nifti", prob=0.2),
    EnsureTyped(keys="nifti"),
])

/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [5]:
from monai.data import CacheDataset
import torch.multiprocessing as mp

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass


ds = CacheDataset(data, transform=transforms, cache_rate=1.0, num_workers=16)




Loading dataset:   0%|          | 0/917 [00:00<?, ?it/s]

Loading dataset: 100%|██████████| 917/917 [08:29<00:00,  1.80it/s]


In [6]:
from monai.data import ThreadDataLoader


loader = ThreadDataLoader(ds, batch_size=100, shuffle=True, num_workers=0)

device = torch.device("cuda")
model = model =ViTAutoEnc(
                    in_channels=1, 
                    patch_size=(16,16,16),
                    img_size=(96,96,96)
                    ).to(device)

opt = torch.optim.Adam(model.parameters(), 1e-3)
loss_fn = torch.nn.MSELoss()


In [7]:
import time
import torch

model.train()
for epoch in range(50):
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.perf_counter()

    running, n_samples = 0.0, 0
    for batch in loader:
        x = batch["nifti"].to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        recon, _ = model(x)
        loss = loss_fn(recon, x)
        loss.backward()
        opt.step()
        running += loss.detach()          # no .item() per step, which would sync every iteration
        n_samples += x.shape[0]

    torch.cuda.synchronize()
    epoch_s = time.perf_counter() - t0
    avg_loss = (running / len(loader)).item()
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    print(f"epoch {epoch:3d}  loss {avg_loss:.4f}  {epoch_s:.1f}s  "
          f"{n_samples / epoch_s:.1f} vol/s  peak {peak_gb:.1f} GB")

epoch   0  loss 0.0528  5.1s  179.6 vol/s  peak 19.3 GB
epoch   1  loss 0.0400  5.5s  166.1 vol/s  peak 19.3 GB
epoch   2  loss 0.0374  5.2s  176.6 vol/s  peak 19.3 GB
epoch   3  loss 0.0320  5.5s  166.5 vol/s  peak 19.3 GB
epoch   4  loss 0.0234  5.5s  168.0 vol/s  peak 19.3 GB
epoch   5  loss 0.0230  5.3s  171.6 vol/s  peak 19.3 GB
epoch   6  loss 0.0186  5.4s  169.1 vol/s  peak 19.3 GB
epoch   7  loss 0.0176  5.4s  169.4 vol/s  peak 19.3 GB
epoch   8  loss 0.0171  5.3s  173.9 vol/s  peak 19.3 GB
epoch   9  loss 0.0168  4.9s  185.4 vol/s  peak 19.3 GB
epoch  10  loss 0.0168  5.5s  165.8 vol/s  peak 19.3 GB
epoch  11  loss 0.0167  5.2s  175.0 vol/s  peak 19.3 GB
epoch  12  loss 0.0168  5.3s  173.6 vol/s  peak 19.3 GB
epoch  13  loss 0.0166  5.1s  179.6 vol/s  peak 19.3 GB
epoch  14  loss 0.0167  5.1s  179.6 vol/s  peak 19.3 GB
epoch  15  loss 0.0166  5.8s  157.0 vol/s  peak 19.3 GB
epoch  16  loss 0.0165  5.6s  165.2 vol/s  peak 19.3 GB
epoch  17  loss 0.0163  5.7s  161.0 vol/s  peak 

In [9]:
print("peak allocated:", torch.cuda.max_memory_allocated() / 1e9, "GB")
print("peak reserved: ", torch.cuda.max_memory_reserved() / 1e9, "GB")
print(torch.cuda.memory_stats()["num_alloc_retries"])

peak allocated: 19.25378048 GB
peak reserved:  28.011659264 GB
0
